In [ ]:
%run ./phase2_common

In [ ]:
REVIEWS = f"{SCHEMA}.silver_movie_reviews"

dbutils.widgets.text("batch_id", "", "Bronze batch ID")
batch_id = dbutils.widgets.get("batch_id").strip()
if not batch_id:
    raise ValueError("batch_id is required")

bronze = spark.table(BRONZE).filter(F.col("batch_id") == batch_id)
if not bronze.limit(1).count():
    raise ValueError(f"No Bronze records found for batch_id={batch_id}")
load_type = bronze.select("load_type").first()[0]

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {REVIEWS} (
    movie_id BIGINT, review_id STRING, reviewer_hash_id STRING, content STRING,
    created_at TIMESTAMP, updated_at TIMESTAMP, rating DOUBLE,
    weeks_since_release INT, source_batch_id STRING, source_file STRING,
    source_extracted_at TIMESTAMP, row_hash STRING, load_timestamp TIMESTAMP
) USING DELTA
""")

In [ ]:
salt = dbutils.secrets.get(scope="tmdb-lakehouse-secrets", key="reviewer-hash-salt")

parsed = (
    bronze.select(
        "movie_id", "batch_id", "source_file", "source_extracted_at",
        F.from_json("raw_payload", MOVIE_SCHEMA).alias("movie"),
    )
    .select("movie_id", "batch_id", "source_file", "source_extracted_at", "movie.*")
)
if parsed.filter(F.col("id").isNull() | (F.col("id") != F.col("movie_id"))).limit(1).count():
    raise ValueError("Some Bronze records have a missing or mismatched movie ID")

exploded = (
    parsed.filter(F.col("reviews.results").isNotNull())
    .select(
        "movie_id", "batch_id", "source_file", "source_extracted_at", "release_date",
        F.explode("reviews.results").alias("r"),
    )
)

identity = F.coalesce(*[
    F.when(F.length(F.trim(c)) > 0, F.trim(c))
    for c in (F.col("r.author_details.username"), F.col("r.author"))
])
scrubbed = F.regexp_replace(
    F.regexp_replace(F.col("r.content"), r"(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", "[EMAIL]"),
    r"(?i)\b(?:https?://|www\.)\S+", "[URL]",
)
created = F.expr("try_to_timestamp(r.created_at)")
release = F.expr("try_to_date(NULLIF(TRIM(release_date), ''), 'yyyy-MM-dd')")

hash_columns = [
    "movie_id", "review_id", "reviewer_hash_id", "content",
    "created_at", "updated_at", "rating", "weeks_since_release",
]
rows = (
    exploded.select(
        "movie_id",
        F.col("r.id").alias("review_id"),
        F.when(identity.isNotNull(), F.sha2(F.concat(F.lit(salt), F.lit(":"), F.lower(identity)), 256))
        .alias("reviewer_hash_id"),
        scrubbed.alias("content"),
        created.alias("created_at"),
        F.expr("try_to_timestamp(r.updated_at)").alias("updated_at"),
        F.col("r.author_details.rating").alias("rating"),
        F.floor(
            (F.unix_timestamp(created) - F.unix_timestamp(release.cast("timestamp"))) / (7 * 24 * 3600)
        ).cast("int").alias("weeks_since_release"),
        F.col("batch_id").alias("source_batch_id"), "source_file", "source_extracted_at",
    )
    .withColumn("row_hash", F.sha2(F.to_json(F.struct(*hash_columns)), 256))
    .withColumn("load_timestamp", F.current_timestamp())
)

In [ ]:
def load():
    if rows.filter(
        F.col("review_id").isNull() | (F.length(F.trim("review_id")) == 0) | F.col("created_at").isNull()
    ).limit(1).count():
        raise ValueError("Reviews with a missing review_id or an invalid created_at found")

    conflicts = (
        rows.groupBy("movie_id", "review_id").agg(F.countDistinct("row_hash").alias("versions"))
        .filter("versions > 1")
    )
    if conflicts.limit(1).count():
        raise ValueError("The same review appears with conflicting content")

    unique = rows.dropDuplicates(["movie_id", "review_id"])
    stored = spark.table(REVIEWS).select(
        "movie_id", "review_id",
        F.col("source_extracted_at").alias("old_ts"), F.col("source_batch_id").alias("old_batch"),
    )
    newer = F.col("source_extracted_at") > F.col("old_ts")
    same_batch = (F.col("source_extracted_at") == F.col("old_ts")) & (F.col("source_batch_id") == F.col("old_batch"))
    eligible = (
        unique.join(stored, ["movie_id", "review_id"], "left")
        .filter(F.col("old_batch").isNull() | newer | same_batch)
        .select(*unique.columns)
    )
    return upsert(REVIEWS, eligible, ["movie_id", "review_id"])


print(run_logged(
    "Bronze-to-Silver", load_type, f"batch_id={batch_id}; table=silver_movie_reviews", batch_id, load,
))